##SparkSession

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.appName('Spark Streaming').getOrCreate()

##SparkStreaming Read

In [0]:
schema = StructType()\
                .add('order_id',StringType())\
                    .add('customer_id',StringType())\
                        .add('Product_Name',StringType())\
                            .add('price',DoubleType())\
                                .add('order_time',TimestampType())

In [0]:
df = spark.readStream\
    .format('csv')\
    .schema(schema)\
    .option('cleanSource','archive')\
    .option('sourceArchiveDir','/Volumes/apachespark/sparkstreaming/p1/archivesrc/')\
    .load('/Volumes/apachespark/sparkstreaming/p1/src/')

##Transformation And ForEachBatch

In [0]:
def transformaing(batch_df,batch_id):

        product_revenue=batch_df.groupBy('Product_Name',window('order_time','10 minutes'))\
                .agg(sum('price').alias('Total_revenue'))
        
        customer_revenue=batch_df.groupBy('customer_id',window('order_time','10 minutes'))\
                .agg(sum('price').alias('Cust_revenue'))
        
        product_revenue.write\
                .format('delta')\
                .mode('append')\
                .save('/Volumes/apachespark/sparkstreaming/p1/sink/productRevenue/')
        
        customer_revenue.write\
                        .format('delta')\
                        .mode('append')\
                        .save('/Volumes/apachespark/sparkstreaming/p1/sink/custRevnue/')


#Writing 

In [0]:
df.writeStream\
    .foreachBatch(transformaing)\
    .trigger(once=True)\
    .option('checkpointLocation','/Volumes/apachespark/sparkstreaming/p1/checkPoint/')\
    .start()

#Customer Based Revenue

In [0]:
%sql
select * from delta.`/Volumes/apachespark/sparkstreaming/p1/sink/custRevnue`

customer_id,window,Cust_revenue
user_5,"List(2026-03-22T10:00:00.000Z, 2026-03-22T10:10:00.000Z)",64000.0
user_1,"List(2026-03-22T10:00:00.000Z, 2026-03-22T10:10:00.000Z)",75000.0
user_2,"List(2026-03-22T10:00:00.000Z, 2026-03-22T10:10:00.000Z)",57500.0
user_3,"List(2026-03-22T10:00:00.000Z, 2026-03-22T10:10:00.000Z)",11000.0
user_4,"List(2026-03-22T10:00:00.000Z, 2026-03-22T10:10:00.000Z)",25000.0
user_2,"List(2026-03-22T10:10:00.000Z, 2026-03-22T10:20:00.000Z)",14000.0
user_3,"List(2026-03-22T10:10:00.000Z, 2026-03-22T10:20:00.000Z)",60000.0
user_1,"List(2026-03-22T10:10:00.000Z, 2026-03-22T10:20:00.000Z)",3500.0
user_5,"List(2026-03-22T10:10:00.000Z, 2026-03-22T10:20:00.000Z)",4000.0
user_4,"List(2026-03-22T10:10:00.000Z, 2026-03-22T10:20:00.000Z)",7500.0


#Product Based Revenue

In [0]:
%sql
select * from delta.`/Volumes/apachespark/sparkstreaming/p1/sink/productRevenue`

Product_Name,window,Total_revenue
shoes,"List(2026-03-22T10:10:00.000Z, 2026-03-22T10:20:00.000Z)",7500.0
phone,"List(2026-03-22T10:10:00.000Z, 2026-03-22T10:20:00.000Z)",14000.0
laptop,"List(2026-03-22T10:10:00.000Z, 2026-03-22T10:20:00.000Z)",60000.0
watch,"List(2026-03-22T10:10:00.000Z, 2026-03-22T10:20:00.000Z)",7500.0
laptop,"List(2026-03-22T10:00:00.000Z, 2026-03-22T10:10:00.000Z)",167000.0
shoes,"List(2026-03-22T10:00:00.000Z, 2026-03-22T10:10:00.000Z)",5500.0
watch,"List(2026-03-22T10:00:00.000Z, 2026-03-22T10:10:00.000Z)",15000.0
phone,"List(2026-03-22T10:00:00.000Z, 2026-03-22T10:10:00.000Z)",45000.0
